# 2025 高教社杯全国大学生数学建模竞赛 A 题

## 烟幕干扰弹的投放策略（圆柱真目标 · 严格全遮蔽）

本仓库对 2025 国赛 A 题采用 **圆柱真目标 + 严格全遮蔽** 建模，并完成问题 1–5 的数值优化与结果输出。

---

## 1. 问题简述

- 假目标位于坐标原点 $O(0,0,0)$；真目标为底面圆心 $(0,200,0)$、半径 $7\,\mathrm{m}$、高 $10\,\mathrm{m}$ 的圆柱体。
- 来袭导弹 $M_1,M_2,M_3$ 以 $300\,\mathrm{m/s}$ 直指假目标。
- 无人机 $FY_1\sim FY_5$ 等高匀速平飞，速度 $v\in[70,140]\,\mathrm{m/s}$。
- 烟幕云团：起爆后半径 $10\,\mathrm{m}$，以 $3\,\mathrm{m/s}$ 匀速下沉，有效时间 $20\,\mathrm{s}$；投放后弹体自由落体。
- 同一无人机相邻投放间隔至少 $1\,\mathrm{s}$（题面：每架投放两枚至少间隔 $1\,\mathrm{s}$）；每架至多 3 枚（问题 5）。

| 导弹 | 初始位置 | 到达假目标时刻 t<sub>hit</sub> (s) |
|:----:|:---------|-------------------------------:|
| M1 | (20000, 0, 2000) | 66.999 |
| M2 | (19000, 600, 2100) | 63.750 |
| M3 | (18000, −600, 1900) | 60.367 |

| 无人机 | 初始位置 |
|--------|----------|
| FY1 | (17800, 0, 1800) |
| FY2 | (12000, 1400, 1400) |
| FY3 | (6000, −3000, 700) |
| FY4 | (11000, 2000, 1800) |
| FY5 | (13000, −2000, 1300) |

---

## 2. 核心模型：圆柱严格全遮蔽

### 2.1 判据

记时刻 $t$ 导弹位置 $M(t)$、云团球心 $C(t)$。对真目标圆柱表面采样点集合 $\{P_i\}$，要求：

$$
\forall i,\quad
d\bigl(C(t),\; \overline{M(t)P_i}\bigr)\le 10
\quad\text{且垂足参数 } s\in[0,1].
$$

即导弹到圆柱表面 **每一个** 采样点的视线段均被烟幕球挡住——**严格全遮蔽**。

### 2.2 表面采样

| 级别 | 点数 | 用途 |
|------|------|------|
| PTS_FAST | ~68 | 粗搜索 |
| PTS_OPT | ~173 | 中等优化 |
| PTS_FINE | ~537 | 终算（推荐报告） |
| PTS_ULTRA | ~1091 | 复核 / 圆周最坏点 |

### 2.3 有效时长

对单枚或多枚烟幕，在时间轴上求并集：

$$
T_{\mathrm{eff}}=\mu\Bigl(\bigcup_k\{t:\;\text{第 }k\text{ 枚云团严格全遮蔽}\}\Bigr).
$$

终算采用高密度时间扫描 + 二分边界细化。

### 2.4 优化框架

- 公共几何：`common/smoke_geom.py`（多导弹 / 多无人机接口）
- 批量评估：NumPy + PyTorch CUDA（RTX 2080 Ti）
- 策略：粗网格 + 随机采样 + 坐标下降；多弹时固定航向/速度、优化投放时刻与引信延时
- 航向角定义：相对 $+x$ 轴逆时针，Excel 中以度表示 $[0,360)$

---

## 3. 结果汇总（圆柱严格全遮蔽）

| 问题 | 场景 | 目标 | 最优有效时长 |
|------|------|------|--------------|
| 问题1 | FY1 固定策略干扰 M1 | 单弹时长 | **1.362100 s** |
| 问题2 | FY1 优化 1 弹干扰 M1 | 单弹时长 | **4.542008 s** |
| 问题3 | FY1 至多 3 弹干扰 M1 | 并集时长 | **7.608993 s** |
| 问题4 | FY1/FY2/FY3 各 1 弹干扰 M1 | 并集时长 | **10.612404 s** |
| 问题5 | FY1–FY5 至多各 3 弹干扰 M1+M2+M3 | 三弹时长之和 | **21.980761 s** |

### 问题5 分导弹

| 导弹 | 并集有效遮蔽 (s) | 主要负责无人机 |
|------|------------------|----------------|
| M1 | 7.609280 | FY1（3 弹） |
| M2 | 10.649877 | FY2（2 弹）+ FY4 + FY3 |
| M3 | 3.721603 | FY5（1 弹） |
| **合计** | **21.980761** | 共 8 枚有效弹 |

---

## 4. 各问策略摘要

### 问题1（给定策略）

- FY1：速度 v = 120 m/s，朝向假目标；投放时刻 t<sub>drop</sub> = 1.5 s，引信延时 τ = 3.6 s。
- 圆柱严格全遮蔽时长：**1.3621 s**（点模型中心约为 1.4055 s，圆柱判据更严格）。

### 问题2（FY1 单弹最优）

| 参数 | 数值 |
|:-----|-----:|
| 航向角 | 176.6187° |
| 速度 | 70 m/s |
| 投放时刻 t<sub>drop</sub> | 0 s |
| 引信延时 τ | 2.4841 s |
| 有效遮蔽时长 | **4.542008 s** |

### 问题3（FY1 三弹）

| 无人机 | 航向角 (°) | 速度 (m/s) | 烟幕弹 | 投放时刻 t<sub>d</sub> (s) | 引信延时 τ (s) |
|:------:|------------:|-----------:|:------:|------------------------------:|----------------:|
| FY1 | 179.6475 | 139.9983 | 1 | 0.0030 | 3.6111 |
| FY1 | 179.6475 | 139.9983 | 2 | 3.7025 | 5.3375 |
| FY1 | 179.6475 | 139.9983 | 3 | 5.5695 | 6.0405 |

三枚烟幕弹的有效遮蔽时间并集：**7.608993 s**。

输出：`附件/result1.xlsx`

### 问题4（三机各一弹 vs M1）

时间窗近似错开，并集接近单弹之和：

| 无人机 | 航向角 (°) | 速度 (m/s) | 投放时刻 t<sub>d</sub> (s) | 引信延时 τ (s) | 单弹有效时长 (s) |
|:------:|------------:|-----------:|------------------------------:|----------------:|------------------:|
| FY1 | 176.6188 | 70.00 | 0.000 | 2.484 | 4.542 |
| FY2 | 306.1909 | 136.85 | 8.551 | 3.986 | 3.836 |
| FY3 | 122.4743 | 92.96 | 31.733 | 7.646 | 2.234 |

三架无人机烟幕弹的有效遮蔽时间并集：**10.612404 s**。

输出：`附件/result2.xlsx`

### 问题5（五机多弹 vs 三导弹）

**分派原则**（由 UAV–导弹配对探针矩阵确定）：

```
        M1     M2     M3
FY1   有效     —      —
FY2   有效   有效     —
FY3   有效   有效     —
FY4    —     有效     —
FY5   有效    —     有效
```

采用：M1←FY1，M2←FY2+FY4+FY3，M3←FY5。

| 无人机 | 目标导弹 | 航向角 (°) | 速度 (m/s) | 烟幕弹数 | 说明 |
|:------:|:--------:|------------:|-----------:|---------:|:-----|
| FY1 | M1 | 179.6475 | 140.0 | 3 | Q3 多弹方案 |
| FY2 | M2 | 293.6618 | 140.0 | 2 | 双弹并集约 7.75 s |
| FY3 | M2 | 86.8031 | 133.4 | 1 | 晚窗补强 |
| FY4 | M2 | 237.6056 | 81.6 | 1 | 中窗补强 |
| FY5 | M3 | 116.3775 | 140.0 | 1 | 唯一有效对 M3 |

输出：`附件/result3.xlsx`

---

## 5. 目录结构

```
2025国赛A题/
├── A题.pdf
├── README.md                 # 本文件
├── common/
│   └── smoke_geom.py         # 公共几何 / 批量核 / 多导弹接口
├── 问题1/ … 问题5/           # 各问脚本与 结果/
├── 附件/
│   ├── result1.xlsx          # 问题3
│   ├── result2.xlsx          # 问题4
│   └── result3.xlsx          # 问题5
├── 点模型对比/
│   ├── point_vs_cylinder_comparison.txt
│   └── 与公开优秀解对比.md  # 与开源/优秀仓数值对照
```

主要结果文件：

| 路径 | 内容 |
|------|------|
| `问题1/结果/q1_cylinder_result.*` | 问题1 |
| `问题2/结果/q2_cylinder_result.*` | 问题2 |
| `问题3/结果/q3_cylinder_result.*` + `result1.xlsx` | 问题3 |
| `问题4/结果/q4_cylinder_result.*` + `result2.xlsx` | 问题4 |
| `问题5/结果/q5_cylinder_result.*` + `result3.xlsx` | 问题5 |
| `点模型对比/与公开优秀解对比.md` | 与公开优秀/开源解对比 |

---

## 6. 运行环境与复现

- Python 3.10（仓库根目录下使用 `python` 或 `py -3.10`）
- 依赖：`numpy`, `torch`（CUDA）, `openpyxl`
- GPU：NVIDIA GeForce RTX 2080 Ti（亦可 CPU，较慢）

在仓库根目录运行示例：

```bat
python -u ".\问题5\q5_cylinder_optimize.py"
```

各问优化脚本位于对应 `问题k/` 目录；几何与判据统一在 `common/smoke_geom.py`。

---

## 7. 说明与讨论

1. **圆柱 vs 点模型**：严格全遮蔽对视线覆盖要求更强，有效时长通常略低于以几何中心为代表的点模型；问题1 已验证两者数量级一致（1.36 s vs 1.41 s）。
2. **问题4 可加性**：三机最优投放落在不相交时间窗，并集 ≈ 单弹之和，说明错峰投放是关键。
3. **问题5 配对可行性**：并非任意 UAV–导弹对都能形成正时长遮蔽；须先做配对探针再分派，否则会出现 M3=0 的无效方案。
4. **多弹收益**：同一无人机多弹主要靠时间轴拼接延长并集；FY1 对 M1 从单弹 4.54 s 提升到三弹 7.61 s；FY2 对 M2 双弹并集约 7.75 s。
5. **数值精度**：终算使用 PTS_FINE + 高密度时间扫描与二分细化；更密采样（ULTRA / 圆周最坏点）与 FINE 差异一般在 $10^{-3}\,\mathrm{s}$ 量级。

---

## 8. 交付物清单

- [x] 问题1–5 圆柱严格全遮蔽数值结果  
- [x] `附件/result1.xlsx`（问题3）  
- [x] `附件/result2.xlsx`（问题4）  
- [x] `附件/result3.xlsx`（问题5）  
- [x] 各问 `q*_cylinder_result.json|txt` 明细  

**推荐写入论文的核心指标（圆柱严格全遮蔽）：**

| 问 | 指标 |
|----|------|
| 1 | 1.3621 s |
| 2 | 4.5420 s |
| 3 | 7.6090 s |
| 4 | 10.6124 s |
| 5 | M1+M2+M3 = **21.9808 s**（7.6093 + 10.6499 + 3.7216） |

---

## 9. 点模型对照（中心点遮蔽）

为比较判据宽松度与计算代价，在相同几何/优化框架下将真目标改为**几何中心点** $(0,200,5)$，判据为

$$
d\bigl(C(t),\; \overline{M(t)\,T}\bigr)\le 10,\quad s\in[0,1],
$$

并完整重跑问题 1–5。结果目录：`点模型对比/`。

### 9.1 遮蔽时长对比

| 问题 | 点模型 (s) | 圆柱严格 (s) | Δ 时长 (s) | 点/圆柱 |
|------|------------|--------------|------------|---------|
| Q1 | 1.405510 | 1.362100 | +0.043 | 1.032× |
| Q2 | 4.810661 | 4.542008 | +0.269 | 1.059× |
| Q3 | 7.748228 | 7.608993 | +0.139 | 1.018× |
| Q4 | 12.841235 | 10.612404 | +2.229 | 1.210× |
| Q5 | **26.851731** | **21.980761** | **+4.871** | **1.222×** |

问题 5 分导弹：

| 导弹 | 点模型 (s) | 圆柱 (s) | Δ (s) |
|------|------------|----------|-------|
| M1 | 7.748 | 7.609 | +0.139 |
| M2 | 13.093 | 10.650 | +2.444 |
| M3 | 6.010 | 3.722 | +2.288 |

### 9.2 运行速度对比

**端到端优化耗时**（同机 RTX 2080 Ti；圆柱为历史完整优化墙钟，点模型为本次重跑）：

| 问题 | 点耗时 (s) | 圆柱耗时 (s) | 加速比 |
|------|------------|--------------|--------|
| Q1 | 0.10 | 446.24 | ~4666× |
| Q2 | 9.17 | 1778.92 | ~194× |
| Q3 | 35.13 | 2658.26 | ~76× |
| Q4 | 20.99 | 1753.23 | ~84× |
| Q5 | 45.59 | 2312.94 | ~51× |

> 说明：端到端加速比同时受判据复杂度与搜索预算影响，不宜单独解读为“核加速比”。

**公平吞吐基准**（同一批 N = 800 个样本、每个样本取 n<sub>t</sub> = 3000 个时间点，仅改变目标采样点数）：

| 评估核 | 耗时 (s) | 相对点模型 |
|--------|----------|------------|
| 点模型（1 点） | 0.014 | 1.0× |
| 圆柱 PTS_FAST（68 点） | 0.261 | 18.6× 更慢 |
| 圆柱 PTS_OPT（173 点） | 1.305 | 93.2× 更慢 |

即：单次并集评估中，圆柱相对点模型约 **19–93 倍** 更慢，与表面采样点数近似成正比。

### 9.3 结论

1. **遮蔽时长**：点模型普遍 ≥ 圆柱严格全遮蔽（判据更松）；Q1–Q3 差距约 1–6%，Q4–Q5 因多机/多弹时间窗拼接，差距扩大到约 20–22%。
2. **运行速度**：点模型优化与评估显著更快；公平核吞吐约 19–93×，端到端约 50–200×（Q1 固定策略更夸张）。
3. **策略一致性**：两者最优航向/速度/投放时刻往往接近；可用点模型快速探路，再用圆柱严格判据终算与稳健性校验。
4. **论文建议**：主文采用圆柱严格全遮蔽（更贴合“遮蔽真目标”）；点模型可作为灵敏度分析附录。

复现（在仓库根目录）：

```bat
python -u ".\run_point_vs_cylinder.py"
```

### 9.4 与公开优秀/开源解对比

与 GitHub 可复现仓库及省级一等奖建模口径的并排对比见：

**`点模型对比/与公开优秀解对比.md`**

要点（注意判据不同，数值不可无说明横比）：

| 问题 | 本项目-圆柱严格 | 公开区间（多源） |
|------|----------------:|------------------|
| Q1 | 1.362 | 1.36–1.45 |
| Q2 | 4.542 | 4.5–4.8 |
| Q3 | 7.609 | 6.4–8.2 |
| Q4 | 10.612 | 11.7–15.5（多含半遮/点判据） |
| Q5 | 21.981 | 20.5–25.4（Julia 38.6 为离群） |

严格全遮天然短于 50% 阈值/中心点方案；Q1–Q2 与社区高度一致。